In [2]:
import scipy.io as sio
import scipy.signal as signal
import numpy as np
import torch

class CSIPreprocessor:
    """
    Préprocessing CSI - Version avec downsampling uniquement
    Format final : (342, 500) - aplati
    """

    def __init__(self, fs=200.0, cutoff=10.0, filter_order=3):
        self.fs = fs
        self.cutoff = cutoff
        self.filter_order = filter_order

        # Pré-calcul filtre Butterworth
        nyq = 0.5 * self.fs
        normal_cutoff = self.cutoff / nyq
        self.b, self.a = signal.butter(self.filter_order, normal_cutoff, btype='low')

    def load_mat(self, filepath):
        """Charge fichier .mat"""
        mat = sio.loadmat(filepath)
        if "CSIamp" not in mat:
            raise KeyError(f"'CSIamp' introuvable dans {filepath}")
        data = mat["CSIamp"]  # (342, 2000)
        return data.astype(np.float32)

    def filter_signal(self, data):
        """Filtrage Butterworth"""
        return signal.filtfilt(self.b, self.a, data, axis=1)

    def downsample(self, data):
        """Downsampling : 2000 → 500 (1 point sur 4)"""
        return data[:, ::4]  # (342, 500)

    def normalize(self, data):
        """Z-score normalization"""
        mean = np.mean(data)
        std = np.std(data)
        if std < 1e-8:
            std = 1e-8
        return (data - mean) / std

    def to_tensor(self, data):
        """Conversion en tenseur PyTorch"""
        return torch.from_numpy(data).float()

    def __call__(self, filepath):
        """Pipeline complet"""
        data = self.load_mat(filepath)   # (342, 2000)
        data = self.filter_signal(data)  # (342, 2000)
        data = self.downsample(data)     # (342, 500) ✅
        data = self.normalize(data)      # (342, 500)
        tensor = self.to_tensor(data)    # (342, 500)
        return tensor

In [4]:
import os
import torch
import numpy as np

# ton preprocesser
# (colle ici ta classe CSIPreprocessor)

# ==========================================
# CONFIG
# ==========================================
DATASET_PATH = "NTU-Fi_HAR"
TRAIN_DIR = os.path.join(DATASET_PATH, "train_amp")
TEST_DIR  = os.path.join(DATASET_PATH, "test_amp")

CLASSES = ["box", "circle", "clean", "fall", "run", "walk"]
label_map = {cls: idx for idx, cls in enumerate(CLASSES)}

preprocessor = CSIPreprocessor(fs=200, cutoff=10, filter_order=3)


# ==========================================
# LOAD ALL FILES FROM A SPLIT
# ==========================================
def load_split(split_path):
    X = []
    y = []

    for class_name in CLASSES:
        class_path = os.path.join(split_path, class_name)

        if not os.path.exists(class_path):
            print(f"⚠️ dossier manquant: {class_path}")
            continue

        files = [f for f in os.listdir(class_path) if f.endswith(".mat")]

        print(f"📂 {class_name} -> {len(files)} fichiers")

        for f in files:
            file_path = os.path.join(class_path, f)

            try:
                tensor = preprocessor(file_path)  # (1, 342, 2000)

                X.append(tensor)
                y.append(label_map[class_name])

            except Exception as e:
                print(f"❌ erreur {file_path}: {e}")

    X = torch.stack(X)  # (N, 1, 342, 2000)
    y = torch.tensor(y, dtype=torch.long)

    return X, y

In [5]:
print("🚀 Loading TRAIN set...")
X_train, y_train = load_split(TRAIN_DIR)

print("🚀 Loading TEST set...")
X_test, y_test = load_split(TEST_DIR)


print("✅ Shape train:", X_train.shape, y_train.shape)
print("✅ Shape test :", X_test.shape, y_test.shape)


# ==========================================
# SAVE (OPTIONNEL MAIS TRÈS IMPORTANT)
# ==========================================
os.makedirs("processed_data", exist_ok=True)

torch.save((X_train, y_train), "processed_data/train.pt")
torch.save((X_test, y_test), "processed_data/test.pt")

print("💾 Dataset sauvegardé dans processed_data/")

🚀 Loading TRAIN set...
📂 box -> 156 fichiers
📂 circle -> 156 fichiers
📂 clean -> 156 fichiers
📂 fall -> 156 fichiers
📂 run -> 156 fichiers
📂 walk -> 156 fichiers
🚀 Loading TEST set...
📂 box -> 44 fichiers
📂 circle -> 44 fichiers
📂 clean -> 44 fichiers
📂 fall -> 44 fichiers
📂 run -> 44 fichiers
📂 walk -> 44 fichiers
✅ Shape train: torch.Size([936, 342, 500]) torch.Size([936])
✅ Shape test : torch.Size([264, 342, 500]) torch.Size([264])
💾 Dataset sauvegardé dans processed_data/
